# 1.2 ReAct Agent

- **对应文档**: [quickstart_agent](https://doc.agentscope.io/tutorial/quickstart_agent.html)
- **本讲目标**: 使用 ReActAgent、理解参数，以及从 AgentBase 自定义 Agent。
- **前置知识**: 01_installation，Message 基本概念。

## 导入与准备

In [ ]:
from agentscope.agent import ReActAgent, AgentBase
from agentscope.formatter import DashScopeChatFormatter
from agentscope.memory import InMemoryMemory
from agentscope.message import Msg
from agentscope.model import DashScopeChatModel
from agentscope.tool import Toolkit, execute_python_code
import asyncio
import os

## 创建 ReAct Agent（需配置 API Key）

以下示例需要设置环境变量 `DASHSCOPE_API_KEY`。仅做结构学习时可先阅读代码，不执行调用。

In [ ]:
async def creating_react_agent() -> None:
    toolkit = Toolkit()
    toolkit.register_tool_function(execute_python_code)

    jarvis = ReActAgent(
        name="Jarvis",
        sys_prompt="You're a helpful assistant named Jarvis",
        model=DashScopeChatModel(
            model_name="qwen-max",
            api_key=os.environ.get("DASHSCOPE_API_KEY", ""),
            stream=True,
            enable_thinking=False,
        ),
        formatter=DashScopeChatFormatter(),
        toolkit=toolkit,
        memory=InMemoryMemory(),
    )

    msg = Msg(name="user", content="Hi! Jarvis, run Hello World in Python.", role="user")
    await jarvis(msg)


# 取消下一行注释并在配置好 API Key 后运行
# asyncio.run(creating_react_agent())

## 从 AgentBase 自定义 Agent

实现 `reply`、`observe`、可选 `handle_interrupt`；ReAct 则还需实现 `_reasoning` 与 `_acting`。

In [ ]:
class MyAgent(AgentBase):
    def __init__(self) -> None:
        super().__init__()
        self.name = "Friday"
        self.sys_prompt = "You're a helpful assistant named Friday."
        self.model = DashScopeChatModel(
            model_name="qwen-max",
            api_key=os.environ.get("DASHSCOPE_API_KEY", ""),
            stream=False,
        )
        self.formatter = DashScopeChatFormatter()
        self.memory = InMemoryMemory()

    async def reply(self, msg: Msg | list[Msg] | None) -> Msg:
        await self.memory.add(msg)
        prompt = await self.formatter.format(
            [Msg("system", self.sys_prompt, "system"), *await self.memory.get_memory()]
        )
        response = await self.model(prompt)
        out = Msg(name=self.name, content=response.content, role="assistant")
        await self.memory.add(out)
        await self.print(out)
        return out

    async def observe(self, msg: Msg | list[Msg] | None) -> None:
        await self.memory.add(msg)

    async def handle_interrupt(self) -> Msg:
        return Msg(self.name, "I noticed you interrupted me, how can I help you?", "assistant")


print("MyAgent 类已定义，可通过 asyncio.run(agent(Msg(...))) 调用")